# Credit Risk PD Model â€” Exploratory Data Analysis

**Dataset:** Home Credit Default Risk (Kaggle)  
**Target:** `TARGET = 1` â†’ payment difficulties (default proxy), `0` â†’ no difficulty

---

## Table of Contents
0. [Setup & Data Loading](#0)
1. [Target Variable](#1)
2. [Missing Values](#2)
3. [Numerical Features â€” Correlation with TARGET](#3)
4. [Categorical Features â€” Cramer's V & Default Rates](#4)
5a. [Demographics â€” Age, Gender, Education](#5a)
5b. [Financial Ratios â€” Credit/Income, Affordability](#5b)
5c. [Employment â€” Tenure, Income Type, Organisation](#5c)
5d. [External Scores â€” EXT_SOURCE 1/2/3](#5d)
5e. [Social Circle & Document Flags](#5e)
6. [Bureau Data Aggregation](#6)
7. [Previous Application EDA](#7)
8. [POS/Cash Balance & Installments](#8)
9. [Credit Card Balance](#9)
10. [WoE / IV Analysis](#10)
11. [Correlation & Multicollinearity](#11)
12. [Summary â€” Top Predictors Shortlist](#12)

## 0. Setup & Data Loading <a id='0'></a>

In [ ]:
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from pathlib import Path
from scipy import stats
from scipy.stats import ks_2samp, chi2_contingency

# Optional WoE/IV â€” install with: pip install scorecardpy
try:
    import scorecardpy as sc
    SCORECARDPY_AVAILABLE = True
    print('scorecardpy loaded OK')
except ImportError:
    SCORECARDPY_AVAILABLE = False
    print('scorecardpy not found â€” WoE/IV section will be skipped. Install: pip install scorecardpy')

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
SEED = 42

In [ ]:
# Paths â€” notebook is inside notebooks/, data is at ../data/data_raw/
ROOT     = Path.cwd().parent
DATA_RAW = ROOT / 'data' / 'data_raw'
FIG_DIR  = Path.cwd() / 'figures'
FIG_DIR.mkdir(exist_ok=True)

APP_TRAIN = DATA_RAW / 'home-credit-default-risk' / 'application_train.csv'
BUREAU    = DATA_RAW / 'bureau.csv'
BB        = DATA_RAW / 'bureau_balance.csv'
PREV_APP  = DATA_RAW / 'previous_application.csv'
POS_CASH  = DATA_RAW / 'POS_CASH_balance.csv'
INSTALL   = DATA_RAW / 'installments_payments.csv'
CC_BAL    = DATA_RAW / 'credit_card_balance.csv'

# Load main application table
app = pd.read_csv(APP_TRAIN)
print(f'application_train shape : {app.shape}')
print(f'Default rate            : {app["TARGET"].mean():.2%}  '
      f'({app["TARGET"].sum():,} defaults / {len(app):,} total)')

## 1. Target Variable <a id='1'></a>

First check: class distribution. Credit datasets are typically heavily imbalanced (~8â€“10% default rate).
This matters for:
- Model training â†’ use `class_weight='balanced'` or `scale_pos_weight`
- Threshold selection â†’ don't use 0.5 default cutoff
- Calibration â†’ raw probabilities must be post-calibrated to the true default rate

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = app['TARGET'].value_counts()
labels = ['0 â€” No difficulty', '1 â€” Difficulty']
colors = ['#2196F3', '#F44336']

axes[0].bar(labels, counts.values, color=colors, edgecolor='white', width=0.5)
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 500, f'{v:,}\n({v/len(app):.1%})',
                 ha='center', va='bottom', fontsize=11)
axes[0].set_title('Target Distribution', fontweight='bold')
axes[0].set_ylabel('Count')
axes[0].set_ylim(0, counts.max() * 1.15)

axes[1].pie(counts.values, labels=labels, colors=colors,
            autopct='%1.1f%%', startangle=90,
            wedgeprops=dict(edgecolor='white', linewidth=2))
axes[1].set_title('Class Balance', fontweight='bold')

plt.suptitle('Target Variable â€” CLASS IMBALANCE', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig(FIG_DIR / '01_target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

scale_pos_weight = (app['TARGET'] == 0).sum() / (app['TARGET'] == 1).sum()
print(f'scale_pos_weight for XGBoost: {scale_pos_weight:.2f}')
print('â†’ Use class_weight="balanced" in LogisticRegression')

## 2. Missing Values <a id='2'></a>

High missingness can signal information (e.g., no car â†’ no car age). Key strategies:
- **< 10% missing** â†’ impute with median/mode, safe to proceed
- **10â€“30% missing** â†’ impute + add binary missingness indicator flag
- **30â€“50% missing** â†’ create missingness flag, impute cautiously
- **> 50% missing** â†’ consider dropping, or use missingness flag only

In [ ]:
miss = (app.isnull().sum() / len(app) * 100).sort_values(ascending=False)
miss = miss[miss > 0]
print(f'Columns with ANY missing  : {len(miss)} / {app.shape[1]}')
print(f'Columns > 50% missing     : {(miss > 50).sum()}')
print(f'Columns > 30% missing     : {(miss > 30).sum()}')
print(f'Columns > 10% missing     : {(miss > 10).sum()}')

fig, ax = plt.subplots(figsize=(10, 12))
miss.head(40).sort_values().plot.barh(ax=ax, color='#607D8B')
ax.axvline(50, color='red',    linestyle='--', linewidth=1.5, label='50% â€” consider drop')
ax.axvline(30, color='orange', linestyle='--', linewidth=1.5, label='30% â€” add flag')
ax.axvline(10, color='green',  linestyle='--', linewidth=1.5, label='10% â€” impute OK')
ax.set_xlabel('% Missing')
ax.set_title('Top 40 Columns by Missing Rate (application_train)', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / '02_missing_values.png', dpi=150)
plt.show()

In [ ]:
# Missing value action table
miss_df = pd.DataFrame({'missing_pct': miss, 'missing_n': app.isnull().sum()[miss.index]})
miss_df['action'] = pd.cut(miss_df['missing_pct'],
                            bins=[-1, 10, 30, 50, 100],
                            labels=['impute_ok', 'impute_caution',
                                    'create_missing_flag', 'consider_drop'])
print('Missing value action summary:')
print(miss_df['action'].value_counts().to_string())
print()
print('Columns to consider dropping (>50% missing):')
print(miss_df[miss_df['missing_pct'] > 50].index.tolist())

In [ ]:
# Information gain: does missingness carry signal?
# For each column with >50% missing, create a binary indicator (1=missing, 0=present)
# and compute mutual information with TARGET to see if absence itself is informative.
from sklearn.feature_selection import mutual_info_classif

high_miss_cols = miss_df[miss_df["missing_pct"] > 50].index.tolist()

miss_indicators = pd.DataFrame(
    {col: app[col].isnull().astype(int) for col in high_miss_cols}
)

# mutual_info_classif expects no NaN — indicators are 0/1 so we are clean
mi_scores = mutual_info_classif(
    miss_indicators, app["TARGET"],
    discrete_features=True, random_state=SEED
)

mi_df = (
    pd.Series(mi_scores, index=high_miss_cols)
    .rename("mutual_information")
    .sort_values(ascending=False)
    .to_frame()
)
mi_df["missing_pct"] = miss_df.loc[high_miss_cols, "missing_pct"]

# Label: meaningful signal if MI > 0.001 (common rule-of-thumb threshold)
mi_df["has_signal"] = mi_df["mutual_information"] > 0.001

print(f"Columns with >50% missing: {len(high_miss_cols)}")
print(f"  → with meaningful MI signal (>0.001): {mi_df['has_signal'].sum()}")
print(f"  → no signal (MI ≤ 0.001): {(~mi_df['has_signal']).sum()}")
print()
print(mi_df.to_string())

# ── Plot ────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, max(4, len(high_miss_cols) * 0.35)))
colors = ["#F44336" if s else "#B0BEC5" for s in mi_df["mutual_information"].sort_values().map(lambda x: x > 0.001)]
mi_df["mutual_information"].sort_values().plot.barh(ax=ax, color=colors)
ax.axvline(0.001, color="black", linestyle="--", linewidth=1.2, label="0.001 signal threshold")
ax.set_xlabel("Mutual Information with TARGET")
ax.set_title(
    "Information Gain of Missingness Indicator vs TARGET
"
    "(columns with >50% missing — red = keep missing flag, grey = no signal)",
    fontweight="bold"
)
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "02b_missing_info_gain.png", dpi=150)
plt.show()


## 3. Numerical Features â€” Correlation with TARGET <a id='3'></a>

**Point-biserial correlation** is equivalent to Pearson's r between a continuous variable and a binary target.
Negative correlation â†’ higher values associate with *less* default (protective factor).

Followed by descriptive stats including skewness â€” highly skewed features may benefit from log-transformation.

In [ ]:
num_cols = app.select_dtypes(include=[np.number]).columns.tolist()
num_cols = [c for c in num_cols if c not in ('SK_ID_CURR', 'TARGET')]
print(f'Numerical columns: {len(num_cols)}')

# Descriptive stats
desc = app[num_cols].describe().T
desc['skewness']    = app[num_cols].skew()
desc['kurtosis']    = app[num_cols].kurtosis()
desc['missing_pct'] = app[num_cols].isnull().mean() * 100

print('\nMost skewed features (top 10):')
print(desc['skewness'].abs().sort_values(ascending=False).head(10).to_string())

In [ ]:
# Point-biserial correlation vs TARGET
pb_corr = {}
for c in num_cols:
    valid = app[[c, 'TARGET']].dropna()
    if len(valid) > 100:
        r, p = stats.pointbiserialr(valid['TARGET'], valid[c])
        pb_corr[c] = {'correlation': r, 'p_value': p}

pb_df = pd.DataFrame(pb_corr).T.sort_values('correlation')
pb_df['abs_corr'] = pb_df['correlation'].abs()
top_num = pb_df.sort_values('abs_corr', ascending=False).head(20)

print('Top 20 numerical predictors (by |correlation|):')
print(top_num['correlation'].to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
sorted_corr = top_num['correlation'].sort_values()
bar_colors  = ['#F44336' if v < 0 else '#2196F3' for v in sorted_corr.values]
sorted_corr.plot.barh(ax=ax, color=bar_colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Top 20 Numerical Features\nPoint-Biserial Correlation with TARGET',
             fontweight='bold')
ax.set_xlabel('Correlation  (negative = higher value â†’ fewer defaults)')
plt.tight_layout()
plt.savefig(FIG_DIR / '03_numerical_correlations.png', dpi=150)
plt.show()

# Show full descriptive stats for top features
desc.loc[top_num.head(10).index, ['mean', 'std', 'min', 'max', 'skewness', 'missing_pct']]

## 4. Categorical Features â€” Cramer's V & Default Rates <a id='4'></a>

**Cramer's V** measures the strength of association between a categorical variable and the binary TARGET.  
It ranges from 0 (no association) to 1 (perfect association) and is derived from the ChiÂ² statistic.

In [ ]:
cat_cols = app.select_dtypes(include=['object']).columns.tolist()
print(f'Categorical columns: {len(cat_cols)}')

chi2_results = {}
for c in cat_cols:
    ct = pd.crosstab(app[c], app['TARGET'])
    if ct.shape[0] > 1:
        chi2, p, dof, _ = chi2_contingency(ct)
        chi2_results[c] = {
            'chi2': chi2, 'p_value': p, 'dof': dof,
            'cramers_v': np.sqrt(chi2 / (len(app) * (min(ct.shape) - 1)))
        }

chi2_df = pd.DataFrame(chi2_results).T.sort_values('cramers_v', ascending=False)
print('\nCategorical predictors ranked by Cramer\'s V:')
print(chi2_df[['cramers_v', 'p_value']].to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
chi2_df['cramers_v'].sort_values(ascending=True).plot.barh(ax=ax, color='#9C27B0')
ax.set_title("Categorical Features â€” Cramer's V with TARGET", fontweight='bold')
ax.set_xlabel("Cramer's V  (0 = no association, 1 = perfect)")
plt.tight_layout()
plt.savefig(FIG_DIR / '04_categorical_association.png', dpi=150)
plt.show()

In [ ]:
# Default rate per category for top 5 categorical features
top_cats = chi2_df.head(5).index.tolist()
fig, axes = plt.subplots(1, len(top_cats), figsize=(20, 5))

portfolio_avg = app['TARGET'].mean() * 100

for ax, col in zip(axes, top_cats):
    dr = app.groupby(col)['TARGET'].agg(['mean', 'count'])
    dr = dr.sort_values('mean', ascending=False).head(15)
    ax.barh(dr.index.astype(str), dr['mean'] * 100,
            color='#E91E63', edgecolor='white')
    ax.axvline(portfolio_avg, color='black',
               linestyle='--', linewidth=1, label=f'Avg {portfolio_avg:.1f}%')
    ax.set_xlabel('Default Rate (%)')
    ax.set_title(col, fontweight='bold', fontsize=9)
    ax.legend(fontsize=8)

plt.suptitle('Default Rate by Top Categorical Features', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / '05_categorical_default_rates.png', dpi=150, bbox_inches='tight')
plt.show()

## 5a. Demographics â€” Age, Gender, Education <a id='5a'></a>

Key demographic signals in credit risk:
- **Age** (`DAYS_BIRTH`): younger borrowers typically have higher default rates
- **Gender** (`CODE_GENDER`): directional differences observed in most portfolios
- **Education** (`NAME_EDUCATION_TYPE`): proxy for income stability and financial literacy
- **Family status**: single / married affects disposable income

In [ ]:
app['AGE_YEARS'] = -app['DAYS_BIRTH'] / 365

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Age distribution by target
for tgt, label, color in [(0, 'No difficulty', '#2196F3'),
                           (1, 'Difficulty',    '#F44336')]:
    axes[0].hist(app.loc[app['TARGET'] == tgt, 'AGE_YEARS'],
                 bins=40, alpha=0.6, label=label, color=color, density=True)
axes[0].set_xlabel('Age (years)')
axes[0].set_ylabel('Density')
axes[0].set_title('Age Distribution by Default Status', fontweight='bold')
axes[0].legend()

# Default rate by age band
app['AGE_BAND'] = pd.cut(app['AGE_YEARS'],
                          bins=[19, 25, 30, 35, 40, 50, 60, 70],
                          labels=['20-25','26-30','31-35','36-40','41-50','51-60','61-70'])
dr_age = app.groupby('AGE_BAND', observed=True)['TARGET'].mean() * 100
dr_age.plot.bar(ax=axes[1], color='#FF7043', edgecolor='white')
axes[1].axhline(app['TARGET'].mean() * 100, color='black',
                linestyle='--', linewidth=1.2, label='Portfolio avg')
axes[1].set_xlabel('Age Band')
axes[1].set_ylabel('Default Rate (%)')
axes[1].set_title('Default Rate by Age Band', fontweight='bold')
axes[1].legend()
axes[1].tick_params(axis='x', rotation=30)

# Gender
dr_gender = app.groupby('CODE_GENDER')['TARGET'].agg(['mean', 'count'])
axes[2].bar(dr_gender.index, dr_gender['mean'] * 100,
            color=['#42A5F5', '#EF5350', '#78909C'], edgecolor='white')
axes[2].axhline(app['TARGET'].mean() * 100, color='black', linestyle='--', linewidth=1.2)
axes[2].set_ylabel('Default Rate (%)')
axes[2].set_title('Default Rate by Gender', fontweight='bold')
for i, (idx, row) in enumerate(dr_gender.iterrows()):
    axes[2].text(i, row['mean'] * 100 + 0.1, f'n={row["count"]:,}',
                 ha='center', fontsize=9)

plt.tight_layout()
plt.savefig(FIG_DIR / '06_demographics.png', dpi=150)
plt.show()

In [ ]:
# Education & Family status
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, col, title in [
    (axes[0], 'NAME_EDUCATION_TYPE', 'Default Rate by Education'),
    (axes[1], 'NAME_FAMILY_STATUS',  'Default Rate by Family Status')
]:
    dr = app.groupby(col)['TARGET'].mean().sort_values(ascending=False) * 100
    dr.plot.bar(ax=ax, color='#5C6BC0', edgecolor='white')
    ax.axhline(app['TARGET'].mean() * 100, color='red',
               linestyle='--', linewidth=1.2, label='Portfolio avg')
    ax.set_ylabel('Default Rate (%)')
    ax.set_title(title, fontweight='bold')
    ax.legend()
    ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig(FIG_DIR / '07_education_family.png', dpi=150)
plt.show()

## 5b. Financial Ratios <a id='5b'></a>

Raw amounts (AMT_CREDIT, AMT_INCOME) are less meaningful than their ratios. Key engineered features:

| Feature | Formula | Interpretation |
|---------|---------|----------------|
| `CREDIT_INCOME_RATIO` | AMT_CREDIT / AMT_INCOME | Leverage â€” higher = more indebted |
| `ANNUITY_INCOME_RATIO` | AMT_ANNUITY / AMT_INCOME | Affordability â€” higher = more stretched |
| `CREDIT_TERM` | AMT_CREDIT / AMT_ANNUITY | Loan tenor in months |
| `GOODS_CREDIT_RATIO` | AMT_GOODS_PRICE / AMT_CREDIT | LTV proxy for consumer loans |

> **Note:** `DAYS_EMPLOYED = 365243` is a sentinel value for pensioners/unemployed â€” must be handled before modelling.

In [ ]:
# DAYS_EMPLOYED anomaly
n_anomaly = (app['DAYS_EMPLOYED'] == 365243).sum()
print(f'DAYS_EMPLOYED == 365243 (sentinel for unemployed/pensioner): {n_anomaly:,} rows ({n_anomaly/len(app):.1%})')

app['DAYS_EMPLOYED_CLEAN']   = app['DAYS_EMPLOYED'].replace(365243, np.nan)
app['FLAG_EMPLOYED_ANOMALY'] = (app['DAYS_EMPLOYED'] == 365243).astype(int)

# Derived financial ratios
app['CREDIT_INCOME_RATIO']  = app['AMT_CREDIT']      / app['AMT_INCOME_TOTAL']
app['ANNUITY_INCOME_RATIO'] = app['AMT_ANNUITY']      / app['AMT_INCOME_TOTAL']
app['CREDIT_TERM']          = app['AMT_CREDIT']       / app['AMT_ANNUITY']
app['GOODS_CREDIT_RATIO']   = app['AMT_GOODS_PRICE']  / app['AMT_CREDIT']
app['INCOME_PER_PERSON']    = app['AMT_INCOME_TOTAL'] / app['CNT_FAM_MEMBERS'].replace(0, np.nan)

ratio_cols = ['CREDIT_INCOME_RATIO', 'ANNUITY_INCOME_RATIO',
              'CREDIT_TERM', 'GOODS_CREDIT_RATIO', 'INCOME_PER_PERSON']

print('\nMedian comparison (TARGET=0 vs TARGET=1):')
for col in ratio_cols:
    med0 = app.loc[app['TARGET'] == 0, col].median()
    med1 = app.loc[app['TARGET'] == 1, col].median()
    print(f'  {col:<28}  no-default={med0:.3f}  default={med1:.3f}  diff={med1-med0:+.3f}')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for ax, col in zip(axes[:5], ratio_cols):
    g0  = app.loc[(app['TARGET'] == 0) & app[col].notna(), col]
    g1  = app.loc[(app['TARGET'] == 1) & app[col].notna(), col]
    cap = app[col].quantile(0.99)
    ax.hist(g0.clip(upper=cap), bins=50, alpha=0.6, color='#2196F3',
            label='No difficulty', density=True)
    ax.hist(g1.clip(upper=cap), bins=50, alpha=0.6, color='#F44336',
            label='Difficulty', density=True)
    ks, p = ks_2samp(g0.clip(upper=cap), g1.clip(upper=cap))
    ax.set_title(f'{col}\nKS={ks:.3f}, p={p:.2e}', fontweight='bold', fontsize=9)
    ax.legend(fontsize=8)
    ax.set_xlabel(col)
    ax.set_ylabel('Density')

# Default rate by number of children
axes[5].cla()
dr_children = (app[app['CNT_CHILDREN'] <= 5]
               .groupby('CNT_CHILDREN')['TARGET'].mean() * 100)
dr_children.plot.bar(ax=axes[5], color='#26A69A', edgecolor='white')
axes[5].axhline(app['TARGET'].mean() * 100, color='red',
                linestyle='--', linewidth=1.2, label='Avg')
axes[5].set_title('Default Rate by No. Children', fontweight='bold')
axes[5].set_ylabel('Default Rate (%)')
axes[5].legend()

plt.suptitle('Financial Ratios â€” Distribution by Default Status (KS test)',
             fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(FIG_DIR / '08_financial_ratios.png', dpi=150, bbox_inches='tight')
plt.show()

## 5c. Employment <a id='5c'></a>

Employment stability is a core credit risk driver:
- Short tenure â†’ less income certainty
- Certain income types (maternity leave, unemployed) have significantly higher default rates
- Organization type carries sector-level risk information

In [ ]:
app['YEARS_EMPLOYED'] = (-app['DAYS_EMPLOYED_CLEAN'] / 365).clip(lower=0)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Employment tenure distribution
for tgt, lbl, col in [(0, 'No difficulty', '#2196F3'),
                       (1, 'Difficulty',    '#F44336')]:
    axes[0].hist(
        app.loc[(app['TARGET'] == tgt) & app['YEARS_EMPLOYED'].notna(),
                'YEARS_EMPLOYED'].clip(upper=40),
        bins=40, alpha=0.6, label=lbl, color=col, density=True)
axes[0].set_xlabel('Years Employed (capped at 40)')
axes[0].set_ylabel('Density')
axes[0].set_title('Employment Tenure by Default Status', fontweight='bold')
axes[0].legend()

# Default rate by income type
dr_inc = app.groupby('NAME_INCOME_TYPE')['TARGET'].mean().sort_values(ascending=False) * 100
dr_inc.plot.bar(ax=axes[1], color='#AB47BC', edgecolor='white')
axes[1].axhline(app['TARGET'].mean() * 100, color='red',
                linestyle='--', linewidth=1.2, label='Portfolio avg')
axes[1].set_title('Default Rate by Income Type', fontweight='bold')
axes[1].set_ylabel('Default Rate (%)')
axes[1].legend()
axes[1].tick_params(axis='x', rotation=30)

# Effect of DAYS_EMPLOYED anomaly flag
dr_flag = app.groupby('FLAG_EMPLOYED_ANOMALY')['TARGET'].mean() * 100
axes[2].bar(['Normal employment', 'Anomalous flag\n(pensioner/unemployed)'],
            dr_flag.values, color=['#42A5F5', '#EF5350'], edgecolor='white')
axes[2].axhline(app['TARGET'].mean() * 100, color='black',
                linestyle='--', linewidth=1.2)
axes[2].set_ylabel('Default Rate (%)')
axes[2].set_title('DAYS_EMPLOYED Anomaly Flag Effect', fontweight='bold')

plt.tight_layout()
plt.savefig(FIG_DIR / '09_employment.png', dpi=150)
plt.show()

In [ ]:
# Default rate by organization type
fig, ax = plt.subplots(figsize=(10, 8))
dr_org = app.groupby('ORGANIZATION_TYPE')['TARGET'].mean().sort_values(ascending=False) * 100
dr_org.head(20).sort_values().plot.barh(ax=ax, color='#FF7043')
ax.axvline(app['TARGET'].mean() * 100, color='black',
           linestyle='--', linewidth=1.2, label='Portfolio avg')
ax.set_title('Default Rate by Organization Type (Top 20 highest risk)', fontweight='bold')
ax.set_xlabel('Default Rate (%)')
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / '10_organization_type.png', dpi=150)
plt.show()

## 5d. External Scores (EXT_SOURCE 1/2/3) <a id='5d'></a>

These are **normalised scores from external data sources** (typically bureau scores or alternative data).
They are usually the **strongest individual predictors** in the application table.

- Higher score â†’ lower default probability (negative correlation with TARGET)
- EXT_SOURCE_1 has ~56% missing â€” need imputation + missingness flag
- Combining them (mean, min) produces even stronger features

In [ ]:
ext_cols = ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']

print('EXT_SOURCE summary:')
for col in ext_cols:
    valid = app[[col, 'TARGET']].dropna()
    r, _  = stats.pointbiserialr(valid['TARGET'], valid[col])
    ks, _ = ks_2samp(valid.loc[valid['TARGET'] == 0, col],
                     valid.loc[valid['TARGET'] == 1, col])
    miss  = app[col].isna().mean()
    print(f'  {col}: corr={r:.4f}  KS={ks:.4f}  missing={miss:.1%}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, col in zip(axes, ext_cols):
    g0 = app.loc[(app['TARGET'] == 0) & app[col].notna(), col]
    g1 = app.loc[(app['TARGET'] == 1) & app[col].notna(), col]
    ax.hist(g0, bins=50, alpha=0.6, color='#2196F3', label='No difficulty', density=True)
    ax.hist(g1, bins=50, alpha=0.6, color='#F44336', label='Difficulty', density=True)
    ks, _ = ks_2samp(g0, g1)
    ax.set_title(f'{col}  (KS={ks:.3f})', fontweight='bold')
    ax.set_xlabel('Score')
    ax.set_ylabel('Density')
    ax.legend()

plt.suptitle('External Credit Scores â€” strongest individual application predictors',
             fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(FIG_DIR / '11_ext_sources.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Pairplot â€” EXT_SOURCE interactions
sample = app[ext_cols + ['TARGET']].dropna().sample(5000, random_state=SEED)
sample['Target'] = sample['TARGET'].map({0: 'No difficulty', 1: 'Difficulty'})

g = sns.pairplot(sample, vars=ext_cols, hue='Target',
                 palette={'No difficulty': '#2196F3', 'Difficulty': '#F44336'},
                 plot_kws=dict(alpha=0.3, s=8), diag_kind='kde')
g.figure.suptitle('EXT_SOURCE Pairplot (n=5,000 sample)', y=1.02, fontweight='bold')
g.figure.savefig(FIG_DIR / '12_ext_source_pairplot.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Combined EXT scores
app['EXT_MEAN'] = app[ext_cols].mean(axis=1)
app['EXT_MIN']  = app[ext_cols].min(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, col, title in [(axes[0], 'EXT_MEAN', 'Mean of EXT_SOURCE 1/2/3'),
                        (axes[1], 'EXT_MIN',  'Min of EXT_SOURCE 1/2/3')]:
    g0 = app.loc[(app['TARGET'] == 0) & app[col].notna(), col]
    g1 = app.loc[(app['TARGET'] == 1) & app[col].notna(), col]
    ks, _ = ks_2samp(g0, g1)
    ax.hist(g0, bins=60, alpha=0.6, color='#2196F3', label='No difficulty', density=True)
    ax.hist(g1, bins=60, alpha=0.6, color='#F44336', label='Difficulty', density=True)
    ax.set_title(f'{title}  (KS={ks:.3f})', fontweight='bold')
    ax.legend()

plt.suptitle('Aggregated External Scores â€” even stronger than individual scores',
             fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / '13_ext_combined.png', dpi=150)
plt.show()

## 5e. Social Circle & Document Flags <a id='5e'></a>

- **Social circle DPD**: defaults in applicant's social network proxy for community risk
- **Document flags**: missing documents can signal fraud risk or data quality issues

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, col in [(axes[0], 'DEF_30_CNT_SOCIAL_CIRCLE'),
                (axes[1], 'DEF_60_CNT_SOCIAL_CIRCLE')]:
    dr = app[app[col] <= 5].groupby(col)['TARGET'].mean() * 100
    dr.plot.bar(ax=ax, color='#26C6DA', edgecolor='white')
    ax.axhline(app['TARGET'].mean() * 100, color='red',
               linestyle='--', linewidth=1.2, label='Portfolio avg')
    ax.set_title(f'Default Rate by {col}', fontweight='bold', fontsize=9)
    ax.set_ylabel('Default Rate (%)')
    ax.legend()
    ax.tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig(FIG_DIR / '14_social_circle.png', dpi=150)
plt.show()

In [ ]:
doc_flag_cols = [c for c in app.columns if c.startswith('FLAG_DOCUMENT')]

doc_dr = {}
for c in doc_flag_cols:
    doc_dr[c] = {
        'dr_flag_1': app.loc[app[c] == 1, 'TARGET'].mean(),
        'dr_flag_0': app.loc[app[c] == 0, 'TARGET'].mean(),
        'lift':      app.loc[app[c] == 1, 'TARGET'].mean() / app['TARGET'].mean()
    }
doc_df = pd.DataFrame(doc_dr).T.sort_values('dr_flag_1', ascending=False)

fig, ax = plt.subplots(figsize=(10, 7))
x = range(len(doc_df))
ax.bar(x, doc_df['dr_flag_1'] * 100, label='Flag=1', color='#EF5350', alpha=0.8)
ax.bar(x, doc_df['dr_flag_0'] * 100, label='Flag=0', color='#42A5F5', alpha=0.5)
ax.axhline(app['TARGET'].mean() * 100, color='black',
           linestyle='--', linewidth=1.2, label='Portfolio avg')
ax.set_xticks(list(x))
ax.set_xticklabels(doc_df.index, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Default Rate (%)')
ax.set_title('Document Flags â€” Default Rate when Flag = 1 vs 0', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / '15_document_flags.png', dpi=150)
plt.show()

print(f'Most impactful document flag: {doc_df["lift"].abs().idxmax()} '
      f'(lift = {doc_df["lift"].abs().max():.2f}x)')
doc_df.sort_values('lift', ascending=False)

## 6. Bureau Data Aggregation <a id='6'></a>

Bureau data records external credit history. We aggregate from `bureau.csv` and `bureau_balance.csv`:

| Engineered Feature | Signal |
|---|---|
| `bureau_max_dpd` | Worst-ever DPD status across all bureau credits |
| `bureau_dpd_rate` | Share of credits with any delinquency |
| `bureau_n_active` | Number of currently open credits |
| `bureau_sum_debt` | Total outstanding bureau debt |
| `bureau_utilization` | Total debt / total credit limit |

In [ ]:
bureau = pd.read_csv(BUREAU)
bb     = pd.read_csv(BB)
print(f'bureau shape        : {bureau.shape}')
print(f'bureau_balance shape: {bb.shape}')

# Map bureau_balance status to numeric DPD severity
status_map = {'C': 0, 'X': -1, '0': 0, '1': 1, '2': 2, '3': 3, '4': 4, '5': 5}
bb['STATUS_NUM'] = bb['STATUS'].map(status_map)

bb_agg = bb.groupby('SK_BUREAU_ID').agg(
    max_dpd_status=('STATUS_NUM', 'max'),
    n_months      =('MONTHS_BALANCE', 'count'),
    n_dpd_months  =('STATUS_NUM', lambda x: (x > 0).sum())
).reset_index()

bureau = bureau.merge(bb_agg, on='SK_BUREAU_ID', how='left')

# Aggregate bureau to one row per SK_ID_CURR
bureau_app = bureau.groupby('SK_ID_CURR').agg(
    bureau_n_credits      =('SK_BUREAU_ID', 'count'),
    bureau_n_active       =('CREDIT_ACTIVE', lambda x: (x == 'Active').sum()),
    bureau_n_closed       =('CREDIT_ACTIVE', lambda x: (x == 'Closed').sum()),
    bureau_max_overdue    =('AMT_CREDIT_MAX_OVERDUE', 'max'),
    bureau_sum_debt       =('AMT_CREDIT_SUM_DEBT', 'sum'),
    bureau_sum_credit     =('AMT_CREDIT_SUM', 'sum'),
    bureau_avg_days_credit=('DAYS_CREDIT', 'mean'),
    bureau_max_dpd        =('max_dpd_status', 'max'),
    bureau_n_dpd_credits  =('n_dpd_months', lambda x: (x > 0).sum()),
    bureau_overdue_current=('AMT_CREDIT_SUM_OVERDUE', 'sum'),
).reset_index()

bureau_app['bureau_dpd_rate']    = (bureau_app['bureau_n_dpd_credits']
                                     / bureau_app['bureau_n_credits'])
bureau_app['bureau_utilization'] = (bureau_app['bureau_sum_debt']
                                     / bureau_app['bureau_sum_credit'].replace(0, np.nan))

app_b = app[['SK_ID_CURR', 'TARGET']].merge(bureau_app, on='SK_ID_CURR', how='left')
print(f'\nBureau features engineered: {len([c for c in bureau_app.columns if c != "SK_ID_CURR"])}')
print(f'Rows with bureau data: {app_b["bureau_n_credits"].notna().sum():,} '
      f'({app_b["bureau_n_credits"].notna().mean():.1%})')

In [ ]:
bureau_feat_cols = [c for c in bureau_app.columns if c != 'SK_ID_CURR']

bureau_ks = {}
for col in bureau_feat_cols:
    valid = app_b[[col, 'TARGET']].dropna()
    if len(valid) > 100:
        ks, _ = ks_2samp(valid.loc[valid['TARGET'] == 0, col],
                          valid.loc[valid['TARGET'] == 1, col])
        bureau_ks[col] = ks

bureau_ks_df = pd.Series(bureau_ks).sort_values(ascending=False)
print('Bureau feature KS statistics (vs TARGET):')
print(bureau_ks_df.to_string())

fig, ax = plt.subplots(figsize=(10, 6))
bureau_ks_df.sort_values().plot.barh(ax=ax, color='#FF7043')
ax.axvline(0.20, color='red',    linestyle='--', linewidth=1.2, label='KS=0.20 (good)')
ax.axvline(0.15, color='orange', linestyle='--', linewidth=1.2, label='KS=0.15 (acceptable)')
ax.set_title('Bureau Aggregated Features â€” KS Statistic vs TARGET', fontweight='bold')
ax.set_xlabel('KS Statistic')
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / '16_bureau_ks.png', dpi=150)
plt.show()

In [ ]:
# Distribution plots for top bureau features
top_bureau = bureau_ks_df.head(4).index.tolist()
fig, axes = plt.subplots(1, len(top_bureau), figsize=(18, 5))

for ax, col in zip(axes, top_bureau):
    g0  = app_b.loc[(app_b['TARGET'] == 0) & app_b[col].notna(), col]
    g1  = app_b.loc[(app_b['TARGET'] == 1) & app_b[col].notna(), col]
    cap = app_b[col].quantile(0.98)
    ax.hist(g0.clip(upper=cap), bins=40, alpha=0.6, color='#2196F3',
            label='No difficulty', density=True)
    ax.hist(g1.clip(upper=cap), bins=40, alpha=0.6, color='#F44336',
            label='Difficulty', density=True)
    ax.set_title(f'{col}\n(KS={bureau_ks_df[col]:.3f})', fontweight='bold', fontsize=9)
    ax.legend(fontsize=8)

plt.suptitle('Top Bureau Features by KS', fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / '17_bureau_top_features.png', dpi=150)
plt.show()

## 7. Previous Application EDA <a id='7'></a>

Previous applications at Home Credit contain rich behavioural information:
- **Refusal rate**: share of past applications that were refused â†’ credit quality signal
- **Number of applications**: frequent applications may signal financial stress
- **Credit amounts**: trend in borrowing behaviour

In [ ]:
prev = pd.read_csv(PREV_APP)
print(f'previous_application shape : {prev.shape}')
print(f'Unique SK_ID_CURR           : {prev["SK_ID_CURR"].nunique():,}')
print('\nContract status breakdown:')
print(prev['NAME_CONTRACT_STATUS'].value_counts().to_string())

In [ ]:
prev['DAYS_DECISION_ABS'] = prev['DAYS_DECISION'].abs()

prev_agg = prev.groupby('SK_ID_CURR').agg(
    prev_n_applications  =('SK_ID_PREV', 'count'),
    prev_n_approved      =('NAME_CONTRACT_STATUS', lambda x: (x == 'Approved').sum()),
    prev_n_refused       =('NAME_CONTRACT_STATUS', lambda x: (x == 'Refused').sum()),
    prev_n_cancelled     =('NAME_CONTRACT_STATUS', lambda x: (x == 'Cancelled').sum()),
    prev_avg_credit      =('AMT_CREDIT', 'mean'),
    prev_max_days_dec    =('DAYS_DECISION_ABS', 'max'),
    prev_avg_days_dec    =('DAYS_DECISION_ABS', 'mean'),
    prev_n_cash          =('NAME_CONTRACT_TYPE', lambda x: (x == 'Cash loans').sum()),
    prev_n_revolving     =('NAME_CONTRACT_TYPE', lambda x: (x == 'Revolving loans').sum()),
).reset_index()

prev_agg['prev_approval_rate'] = prev_agg['prev_n_approved'] / prev_agg['prev_n_applications']
prev_agg['prev_refusal_rate']  = prev_agg['prev_n_refused']  / prev_agg['prev_n_applications']

app_p    = app[['SK_ID_CURR', 'TARGET']].merge(prev_agg, on='SK_ID_CURR', how='left')
prev_num = [c for c in prev_agg.columns if c != 'SK_ID_CURR']

prev_ks = {}
for col in prev_num:
    valid = app_p[[col, 'TARGET']].dropna()
    if len(valid) > 100:
        ks, _ = ks_2samp(valid.loc[valid['TARGET'] == 0, col],
                          valid.loc[valid['TARGET'] == 1, col])
        prev_ks[col] = ks

prev_ks_df = pd.Series(prev_ks).sort_values(ascending=False)
print('Previous application KS statistics (vs TARGET):')
print(prev_ks_df.to_string())

fig, ax = plt.subplots(figsize=(10, 6))
prev_ks_df.sort_values().plot.barh(ax=ax, color='#26A69A')
ax.set_title('Previous Application Features â€” KS Statistic vs TARGET', fontweight='bold')
ax.set_xlabel('KS Statistic')
plt.tight_layout()
plt.savefig(FIG_DIR / '18_prev_app_ks.png', dpi=150)
plt.show()

In [ ]:
# Refusal rate distribution â€” key behavioural signal
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, col, title in [
    (axes[0], 'prev_refusal_rate',  'Previous Refusal Rate'),
    (axes[1], 'prev_approval_rate', 'Previous Approval Rate')
]:
    g0 = app_p.loc[(app_p['TARGET'] == 0) & app_p[col].notna(), col]
    g1 = app_p.loc[(app_p['TARGET'] == 1) & app_p[col].notna(), col]
    ks, _ = ks_2samp(g0, g1)
    ax.hist(g0, bins=30, alpha=0.6, color='#2196F3', label='No difficulty', density=True)
    ax.hist(g1, bins=30, alpha=0.6, color='#F44336', label='Difficulty',    density=True)
    ax.set_title(f'{title}\nKS={ks:.3f}', fontweight='bold')
    ax.set_xlabel(col)
    ax.legend()

plt.tight_layout()
plt.savefig(FIG_DIR / '19_prev_refusal_approval.png', dpi=150)
plt.show()

## 8. POS/Cash Balance & Installments <a id='8'></a>

Payment history is the strongest behavioural predictor. From installments we can derive:
- **inst_late_rate**: proportion of installments paid late
- **inst_max_days_late**: worst payment delay ever observed
- **inst_total_underpay**: cumulative underpayment amount

In [ ]:
pos  = pd.read_csv(POS_CASH)
inst = pd.read_csv(INSTALL)
print(f'POS_CASH_balance shape     : {pos.shape}')
print(f'installments_payments shape: {inst.shape}')

# Payment behaviour engineering
inst['PAYMENT_DIFF'] = inst['AMT_INSTALMENT'] - inst['AMT_PAYMENT']
inst['DAYS_LATE']    = (inst['DAYS_ENTRY_PAYMENT'] - inst['DAYS_INSTALMENT']).clip(lower=0)
inst['LATE_FLAG']    = (inst['DAYS_LATE'] > 0).astype(int)

inst_agg = inst.groupby('SK_ID_CURR').agg(
    inst_n_instalments =('NUM_INSTALMENT_NUMBER', 'count'),
    inst_max_days_late =('DAYS_LATE', 'max'),
    inst_avg_days_late =('DAYS_LATE', 'mean'),
    inst_n_late        =('LATE_FLAG', 'sum'),
    inst_max_underpay  =('PAYMENT_DIFF', 'max'),
    inst_total_underpay=('PAYMENT_DIFF', lambda x: x.clip(lower=0).sum()),
).reset_index()
inst_agg['inst_late_rate'] = inst_agg['inst_n_late'] / inst_agg['inst_n_instalments']

app_i    = app[['SK_ID_CURR', 'TARGET']].merge(inst_agg, on='SK_ID_CURR', how='left')
inst_num = [c for c in inst_agg.columns if c != 'SK_ID_CURR']

inst_ks = {}
for col in inst_num:
    valid = app_i[[col, 'TARGET']].dropna()
    if len(valid) > 100:
        ks, _ = ks_2samp(valid.loc[valid['TARGET'] == 0, col],
                          valid.loc[valid['TARGET'] == 1, col])
        inst_ks[col] = ks

inst_ks_df = pd.Series(inst_ks).sort_values(ascending=False)
print('\nInstallment feature KS statistics:')
print(inst_ks_df.to_string())

fig, ax = plt.subplots(figsize=(10, 5))
inst_ks_df.sort_values().plot.barh(ax=ax, color='#8D6E63')
ax.set_title('Installment Payment Features â€” KS vs TARGET', fontweight='bold')
ax.set_xlabel('KS Statistic')
plt.tight_layout()
plt.savefig(FIG_DIR / '20_installment_ks.png', dpi=150)
plt.show()

In [ ]:
# Late rate and max late days distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, col, title in [
    (axes[0], 'inst_late_rate',    'Installment Late Rate'),
    (axes[1], 'inst_max_days_late','Max Days Late on Installment')
]:
    g0  = app_i.loc[(app_i['TARGET'] == 0) & app_i[col].notna(), col]
    g1  = app_i.loc[(app_i['TARGET'] == 1) & app_i[col].notna(), col]
    cap = app_i[col].quantile(0.98)
    ks, _ = ks_2samp(g0.clip(upper=cap), g1.clip(upper=cap))
    ax.hist(g0.clip(upper=cap), bins=40, alpha=0.6, color='#2196F3',
            label='No difficulty', density=True)
    ax.hist(g1.clip(upper=cap), bins=40, alpha=0.6, color='#F44336',
            label='Difficulty', density=True)
    ax.set_title(f'{title}\nKS={ks:.3f}', fontweight='bold')
    ax.legend()

plt.tight_layout()
plt.savefig(FIG_DIR / '21_installment_dist.png', dpi=150)
plt.show()

## 9. Credit Card Balance <a id='9'></a>

Credit card utilization and payment behaviour are strong behavioural signals:
- High utilization â†’ financially stretched
- DPD months on credit card â†’ direct delinquency history

In [ ]:
cc = pd.read_csv(CC_BAL)
print(f'credit_card_balance shape : {cc.shape}')

cc['UTILIZATION']    = cc['AMT_BALANCE'] / cc['AMT_CREDIT_LIMIT_ACTUAL'].replace(0, np.nan)
cc['PAYMENT_RATIO']  = cc['AMT_PAYMENT_CURRENT'] / cc['AMT_INSTALMENT'].replace(0, np.nan)

cc_agg = cc.groupby('SK_ID_CURR').agg(
    cc_n_months        =('MONTHS_BALANCE', 'count'),
    cc_avg_balance     =('AMT_BALANCE', 'mean'),
    cc_max_balance     =('AMT_BALANCE', 'max'),
    cc_avg_utilization =('UTILIZATION', 'mean'),
    cc_max_utilization =('UTILIZATION', 'max'),
    cc_max_dpd         =('SK_DPD', 'max'),
    cc_n_dpd           =('SK_DPD', lambda x: (x > 0).sum()),
    cc_avg_drawings    =('AMT_DRAWINGS_CURRENT', 'mean'),
).reset_index()

app_c  = app[['SK_ID_CURR', 'TARGET']].merge(cc_agg, on='SK_ID_CURR', how='left')
cc_num = [c for c in cc_agg.columns if c != 'SK_ID_CURR']

cc_ks = {}
for col in cc_num:
    valid = app_c[[col, 'TARGET']].dropna()
    if len(valid) > 100:
        ks, _ = ks_2samp(valid.loc[valid['TARGET'] == 0, col],
                          valid.loc[valid['TARGET'] == 1, col])
        cc_ks[col] = ks

cc_ks_df = pd.Series(cc_ks).sort_values(ascending=False)
print('\nCredit card feature KS statistics:')
print(cc_ks_df.to_string())

fig, ax = plt.subplots(figsize=(9, 5))
cc_ks_df.sort_values().plot.barh(ax=ax, color='#EF9A9A')
ax.set_title('Credit Card Features â€” KS vs TARGET', fontweight='bold')
ax.set_xlabel('KS Statistic')
plt.tight_layout()
plt.savefig(FIG_DIR / '22_cc_balance_ks.png', dpi=150)
plt.show()

## 10. WoE / IV Analysis <a id='10'></a>

**Weight of Evidence (WoE)** encodes the log-odds of default within each bin of a feature.  
**Information Value (IV)** summarises total predictive power:

| IV Range | Interpretation |
|----------|----------------|
| < 0.02   | Useless â€” exclude |
| 0.02â€“0.10 | Weak â€” use with caution |
| 0.10â€“0.30 | Medium â€” good candidate |
| > 0.30   | Strong â€” high predictive power |

> Requires `scorecardpy`: `pip install scorecardpy`

In [ ]:
if SCORECARDPY_AVAILABLE:
    woe_num_cols = [
        'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3',
        'DAYS_BIRTH', 'DAYS_EMPLOYED_CLEAN',
        'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY',
        'CREDIT_INCOME_RATIO', 'ANNUITY_INCOME_RATIO',
        'DAYS_ID_PUBLISH', 'DAYS_REGISTRATION', 'DAYS_LAST_PHONE_CHANGE',
        'AMT_REQ_CREDIT_BUREAU_YEAR', 'AMT_REQ_CREDIT_BUREAU_MON',
        'OWN_CAR_AGE', 'CNT_CHILDREN', 'CNT_FAM_MEMBERS', 'REGION_RATING_CLIENT',
    ]
    woe_cat_cols = [
        'NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY',
        'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE',
        'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'WEEKDAY_APPR_PROCESS_START',
    ]
    woe_cols = [c for c in woe_num_cols + woe_cat_cols if c in app.columns]
    woe_df   = app[woe_cols + ['TARGET']].copy()

    bins = sc.woebin(woe_df, y='TARGET', x=woe_cols, positive='bad|1', print_info=False)

    iv_vals = {feat: bins[feat]['bin_iv'].sum() for feat in bins}
    iv_df   = pd.Series(iv_vals).sort_values(ascending=False)

    print('Information Value summary:')
    print(iv_df.to_string())
else:
    print('scorecardpy not installed â€” install with: pip install scorecardpy')

In [ ]:
if SCORECARDPY_AVAILABLE:
    fig, ax = plt.subplots(figsize=(10, 8))
    bar_c = [
        '#F44336' if v > 0.3 else
        '#FF9800' if v > 0.1 else
        '#4CAF50' if v > 0.02 else '#9E9E9E'
        for v in iv_df.sort_values().values
    ]
    iv_df.sort_values().plot.barh(ax=ax, color=bar_c)
    ax.axvline(0.02, color='gray',   linestyle='--', linewidth=1, label='Weak (0.02)')
    ax.axvline(0.1,  color='orange', linestyle='--', linewidth=1, label='Medium (0.10)')
    ax.axvline(0.3,  color='red',    linestyle='--', linewidth=1, label='Strong (0.30)')
    ax.set_title('Information Value (IV) by Feature', fontweight='bold')
    ax.set_xlabel('IV')
    ax.legend(loc='lower right')
    plt.tight_layout()
    plt.savefig(FIG_DIR / '23_iv_summary.png', dpi=150)
    plt.show()

In [ ]:
if SCORECARDPY_AVAILABLE:
    # WoE plots for top 4 features
    top_iv_feats = iv_df.head(4).index.tolist()
    fig, axes = plt.subplots(1, len(top_iv_feats), figsize=(20, 5))

    for ax, feat in zip(axes, top_iv_feats):
        bin_df = bins[feat].copy()
        bin_df['bin'] = bin_df['bin'].astype(str)
        ax.bar(range(len(bin_df)), bin_df['woe'], color='#7986CB', edgecolor='white')
        ax.axhline(0, color='black', linewidth=0.8)
        ax.set_xticks(range(len(bin_df)))
        ax.set_xticklabels(bin_df['bin'], rotation=45, ha='right', fontsize=7)
        ax.set_title(f'{feat}\nIV={iv_df[feat]:.4f}', fontweight='bold', fontsize=9)
        ax.set_ylabel('WoE')

    plt.suptitle('WoE Plots â€” Top 4 Features by IV', fontweight='bold')
    plt.tight_layout()
    plt.savefig(FIG_DIR / '24_woe_top_features.png', dpi=150, bbox_inches='tight')
    plt.show()

## 11. Correlation & Multicollinearity <a id='11'></a>

High feature correlations (|r| > 0.70) indicate **multicollinearity** â€” problematic for logistic regression  
(inflated standard errors, unstable coefficients). Options:
- Drop one of the pair (prefer the one with higher KS)
- Use PCA to combine correlated features
- Regularise strongly (Ridge/L2)

Note: `AGE_YEARS` and `DAYS_BIRTH` are perfectly collinear â€” keep only one.

In [ ]:
key_num = [
    'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3',
    'DAYS_BIRTH', 'AGE_YEARS',
    'DAYS_EMPLOYED_CLEAN', 'YEARS_EMPLOYED',
    'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY',
    'CREDIT_INCOME_RATIO', 'ANNUITY_INCOME_RATIO', 'CREDIT_TERM',
    'DAYS_ID_PUBLISH', 'DAYS_REGISTRATION',
    'EXT_MEAN', 'EXT_MIN',
    'TARGET'
]
key_num = [c for c in key_num if c in app.columns]

corr_matrix = app[key_num].corr()

fig, ax = plt.subplots(figsize=(14, 12))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.5, cbar_kws={'shrink': 0.7},
            ax=ax, annot_kws={'size': 8})
ax.set_title('Correlation Matrix â€” Key Numerical Features', fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / '25_correlation_matrix.png', dpi=150)
plt.show()

In [ ]:
# Flag high-correlation pairs (potential multicollinearity)
corr_no_tgt = app[[c for c in key_num if c != 'TARGET']].corr()

print('Highly correlated feature pairs (|r| > 0.70):')
high_corr_pairs = []
for i in range(len(corr_no_tgt.columns)):
    for j in range(i + 1, len(corr_no_tgt.columns)):
        r  = corr_no_tgt.iloc[i, j]
        c1 = corr_no_tgt.columns[i]
        c2 = corr_no_tgt.columns[j]
        if abs(r) > 0.70:
            high_corr_pairs.append({'feature_1': c1, 'feature_2': c2, 'r': round(r, 3)})
            print(f'  {c1}  x  {c2}  ->  r={r:.3f}')

if not high_corr_pairs:
    print('  No pairs with |r| > 0.70 in this set')

## 12. Summary â€” Top Predictors Shortlist <a id='12'></a>

Rank all engineered features across all tables by KS statistic to identify the best candidates for modelling.

In [ ]:
all_ks = {}

# Application numericals
for col in num_cols:
    valid = app[[col, 'TARGET']].dropna()
    if len(valid) > 100:
        ks, _ = ks_2samp(valid.loc[valid['TARGET'] == 0, col],
                          valid.loc[valid['TARGET'] == 1, col])
        all_ks[f'app__{col}'] = ks

# Engineered ratios
for col in ['EXT_MEAN', 'EXT_MIN', 'CREDIT_INCOME_RATIO',
            'ANNUITY_INCOME_RATIO', 'CREDIT_TERM',
            'AGE_YEARS', 'YEARS_EMPLOYED']:
    if col in app.columns:
        valid = app[[col, 'TARGET']].dropna()
        if len(valid) > 100:
            ks, _ = ks_2samp(valid.loc[valid['TARGET'] == 0, col],
                              valid.loc[valid['TARGET'] == 1, col])
            all_ks[f'app_eng__{col}'] = ks

# Bureau
for col, ks in bureau_ks.items():
    all_ks[f'bureau__{col}'] = ks

# Previous application
for col, ks in prev_ks.items():
    all_ks[f'prev__{col}'] = ks

# Installments
for col, ks in inst_ks.items():
    all_ks[f'inst__{col}'] = ks

# Credit card
for col, ks in cc_ks.items():
    all_ks[f'cc__{col}'] = ks

ks_summary = pd.Series(all_ks).sort_values(ascending=False)
print('Top 30 features across all tables (KS statistic):')
print(ks_summary.head(30).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))
top30 = ks_summary.head(30).sort_values()
bar_c = [
    '#F44336' if v > 0.20 else
    '#FF9800' if v > 0.15 else '#4CAF50'
    for v in top30.values
]
top30.plot.barh(ax=ax, color=bar_c)
ax.axvline(0.20, color='red',    linestyle='--', linewidth=1.5, label='KS=0.20 (good)')
ax.axvline(0.15, color='orange', linestyle='--', linewidth=1.5, label='KS=0.15 (acceptable)')
ax.set_title('Top 30 Features (All Tables) â€” KS Statistic vs TARGET', fontweight='bold')
ax.set_xlabel('KS Statistic')
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / '26_top_features_overall.png', dpi=150)
plt.show()

In [ ]:
print("""
====================================================================
EDA COMPLETE â€” KEY FINDINGS
====================================================================

1.  CLASS IMBALANCE: ~8% default rate
    â†’ class_weight='balanced'  or  scale_pos_weight â‰ˆ 11 in XGBoost

2.  STRONGEST INDIVIDUAL PREDICTORS (application table)
    - EXT_SOURCE_2, EXT_SOURCE_3, EXT_SOURCE_1  (external bureau scores)
    - EXT_MEAN / EXT_MIN (combined even stronger)
    - DAYS_BIRTH / AGE_YEARS      (younger â†’ higher default rate)
    - DAYS_EMPLOYED               (shorter tenure â†’ higher risk)
    - DAYS_ID_PUBLISH             (recent ID change = risk flag)
    - ANNUITY_INCOME_RATIO        (affordability metric)

3.  BUREAU: bureau_max_dpd, bureau_dpd_rate are highly predictive
    Clients with any prior delinquency default significantly more.

4.  INSTALLMENTS: inst_max_days_late, inst_late_rate
    Strong behavioural signals from payment history.

5.  PREVIOUS APPS: prev_refusal_rate is a key risk signal.

6.  MISSING VALUES TO HANDLE:
    - EXT_SOURCE_1 (~56% missing)  â†’ impute median + FLAG_EXT1_MISSING
    - OWN_CAR_AGE  (~66% missing)  â†’ FLAG_OWN_CAR + impute
    - Building features            â†’ consider dropping or flagging

7.  ANOMALIES:
    - DAYS_EMPLOYED == 365243      â†’ replace NaN + FLAG_EMPLOYED_ANOMALY

8.  MULTICOLLINEARITY:
    - DAYS_BIRTH / AGE_YEARS       â†’ keep only AGE_YEARS
    - DAYS_EMPLOYED / YEARS_EMPLOYED â†’ keep only YEARS_EMPLOYED
    - EXT_SOURCE_* highly correlated â†’ use EXT_MEAN as summary feature

Next step: 02_feature_engineering.py
""")